In [2]:
import pandas as pd
import numpy as np
import ast

In [3]:
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

In [4]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [5]:
credits.head(1)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [6]:
#merging both the dataset 
data = movies.merge(credits , on = 'title')

In [7]:
data.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count', 'movie_id', 'cast', 'crew'],
      dtype='str')

In [8]:
revised_data = data[["movie_id" , "title" , "overview" , "genres" , "keywords" , "cast" , "crew"]]

In [9]:
revised_data.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [10]:
revised_data.dropna(inplace=True)

In [11]:
revised_data.isnull().sum()

movie_id    0
title       0
overview    0
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [12]:
revised_data.iloc[0].genres

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [13]:
def convert(obj):
    l = []
    for i in ast.literal_eval(obj):
        l.append(i['name'])
    return l

In [14]:
revised_data["genres"] = revised_data["genres"].apply(convert)

In [15]:
revised_data["keywords"] = revised_data["keywords"].apply(convert)

In [16]:
def convert3(obj):
    l = []
    count = 0
    for i in ast.literal_eval(obj):
        if count == 3:
            break
        l.append(i['name'])
        count += 1
    return l

In [17]:
revised_data["cast"] = revised_data["cast"].apply(convert3)

In [18]:
def fetch_dir(obj):
    l = []

    for i in ast.literal_eval(obj):
        if i["job"] == "Director":
            l.append(i["name"])
            break 
    return l

In [19]:
revised_data["crew"] = revised_data["crew"].apply(fetch_dir)

In [20]:
revised_data["overview"] = revised_data["overview"].apply(lambda x:x.split())

In [21]:
revised_data["genres"] = revised_data["genres"].apply(lambda x:[i.replace(" " , "") for i in x])
revised_data["keywords"] = revised_data["keywords"].apply(lambda x:[i.replace(" " , "") for i in x])
revised_data["cast"] = revised_data["cast"].apply(lambda x:[i.replace(" " , "") for i in x])
revised_data["crew"] = revised_data["crew"].apply(lambda x:[i.replace(" " , "") for i in x])

In [23]:
revised_data["tags"] = revised_data["overview"] + revised_data["genres"] + revised_data["keywords"] + revised_data["cast"] + revised_data["crew"]

In [24]:
revised_data.head(1)

,movie_id,title,overview,genres,keywords,cast,crew,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."


In [25]:
final_data = revised_data[["movie_id" , "title" , "tags"]]

In [26]:
final_data.head(1)

,movie_id,title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."


In [38]:
import nltk
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [47]:
def stem(text):
    l = []

    for i in text.split():
        l.append(ps.stem(i))

    return " ".join(l)

In [28]:
final_data["tags"] = final_data["tags"].apply(lambda x:" ".join(x))

In [32]:
final_data["tags"][0]

'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Action Adventure Fantasy ScienceFiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d SamWorthington ZoeSaldana SigourneyWeaver JamesCameron'

In [33]:
final_data["tags"] = final_data["tags"].apply(lambda x:x.lower())

In [50]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000 , stop_words='english')

In [51]:
vector = cv.fit_transform(final_data["tags"]).toarray()

In [49]:
final_data["tags"] = final_data["tags"].apply(stem)

In [58]:
cv.get_feature_names_out()

array(['000', '007', '10', ..., 'zone', 'zoo', 'zooeydeschanel'],
      shape=(5000,), dtype=object)

In [59]:
from sklearn.metrics.pairwise import cosine_similarity

In [62]:
similarity = cosine_similarity(vector)

In [66]:
sorted(list(enumerate(similarity[0])),reverse = True , key = lambda x:x[1])[1:6]

[(1214, np.float64(0.28676966733820225)),
 (2405, np.float64(0.26901379342448517)),
 (3728, np.float64(0.2605130246476754)),
 (507, np.float64(0.255608593705383)),
 (539, np.float64(0.25038669783359574))]

In [69]:
def recomend(movie):
    movie_index = final_data[final_data["title"] == movie].index[0]
    distance = similarity[movie_index]
    movies_list = sorted(list(enumerate(distance)),reverse = True , key = lambda x:x[1])[1:6]

    for i in movies_list:
        print(final_data.iloc[i[0]].title)
    return

In [72]:
recomend("Avatar")

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.


In [73]:
import pickle
pickle.dump(final_data , open("movies_data.pkl" , "wb"))

In [74]:
pickle.dump(similarity , open("similarity.pkl" , "wb"))